# Лекция 1. Введение: постановка задачи, классы задач, примеры

*Вычислительная оптимизация, магистратура, 1 курс. 9 сентября 2026.*

**План лекции**

1. Зачем это вам: три задачи
2. Три ингредиента и два этапа
3. Стандартная форма и обозначения
4. Основные определения: допустимое множество, локальные и глобальные минимумы
5. Когда решение существует
6. Классы задач
7. Четыре примера
8. Что значит «решить задачу численно»
9. Итоги лекции
10. Обзор курса, инструменты, литература
11. Практика на занятии
12. Упражнения

Конспект опирается на главу 1 лекционных заметок М. Диля (*Lecture Notes on Numerical Optimization*, 2016) и главу 1 книги Nocedal & Wright. Код демонстраций — ноутбук [`demo01.ipynb`](demo01.ipynb): пошагово, с пояснениями по `scipy.optimize` и стандартной формой каждого примера. Картинки конспекта строятся скриптом [`make_figures.py`](make_figures.py).

---

## 1. Зачем это вам: три задачи

Прежде чем давать определения, посмотрим на три задачи. Первые две вернутся в лекциях 3 и 5, третья — подвешенная цепь — будет сопровождать нас весь курс.

| Подогнать кривую к данным | Спланировать производство | Найти форму подвешенной цепи |
|:---:|:---:|:---:|
| ![МНК](img/07_lstsq.png) | ![LP](img/08_lp.png) | ![цепь](img/09_chain.png) |
| По 40 зашумлённым точкам найти коэффициенты полинома | Сколько выпускать каждого продукта, чтобы прибыль была максимальной при ограниченных мощностях | Где окажутся грузы на пружинах, если их подвесить (и что изменится, если снизу пол) |

Все три задачи выглядят по-разному, но устроены одинаково: есть **что выбирать**, есть **что считать хорошим** и есть **чего нельзя нарушать**. Курс — о том, как записать это единообразно и как затем найти ответ на компьютере.

## 2. Три ингредиента и два этапа

Оптимизация — выбор **наилучшего решения** из множества **допустимых**. Любая задача оптимизации состоит из трёх ингредиентов:

- **переменные решения** (decision variables) $x \in \mathbb{R}^n$ — то, чем мы управляем;
- **целевая функция** (objective) $f(x)$ — числовой критерий качества, который мы минимизируем (или максимизируем);
- **ограничения** (constraints) — условия, которые обязано выполнять решение.

Работа с задачей всегда состоит из двух этапов:

1. **Моделирование**: перевод содержательной постановки в математическую — выбор переменных, критерия, ограничений. От решений на этом этапе зависит, насколько задача будет трудной.
2. **Решение**: применение численного алгоритма. Этому в основном посвящён курс, но мы будем постоянно возвращаться к моделированию: одна и та же задача, записанная по-разному, может решаться за миллисекунды или не решаться вовсе.

Где встречается: управление (привести систему в цель с минимальными затратами), оценивание и подгонка моделей, машинное обучение (обучение — минимизация функции потерь), экономика и логистика (планирование, портфели, маршруты), инженерное проектирование (минимальная масса при ограничениях на прочность).

## 3. Стандартная форма и обозначения

### 3.1. Задача нелинейного программирования

Основной объект курса — задача **нелинейного программирования** (nonlinear program, NLP):

$$
\min_{x \in \mathbb{R}^n} \; f(x)
\quad \text{при} \quad
g(x) = 0, \qquad h(x) \ge 0 .
\tag{NLP}
$$

Здесь $f:\mathbb{R}^n \to \mathbb{R}$ — целевая функция, $g:\mathbb{R}^n \to \mathbb{R}^{m}$ задаёт ограничения-равенства, $h:\mathbb{R}^n \to \mathbb{R}^{p}$ — ограничения-неравенства (покомпонентно). Если не оговорено иное, все функции считаем дважды непрерывно дифференцируемыми ($C^2$).

Такая форма не ограничивает общности — любую задачу можно к ней привести:

- максимизация $\max f(x)$ — это $\min (-f(x))$;
- неравенство $h(x) \le 0$ — это $-h(x) \ge 0$;
- ограничение $a \le x_i \le b$ — это два неравенства;
- неравенство $h(x) \ge 0$ можно превратить в равенство с помощью **вспомогательной переменной** (slack): $h(x) - s = 0,\; s \ge 0$.

Вспомогательные переменные — главный инструмент моделирования; в разделе 8 мы увидим, как с их помощью избавиться от модуля.

> **Замечание о соглашениях.** В разных книгах знаки различаются: у Nocedal & Wright и Диля неравенства пишутся как $c_i(x) \ge 0$, у Boyd & Vandenberghe — как $f_i(x) \le 0$. Мы придерживаемся формы (NLP) с $h(x) \ge 0$.

### 3.2. Обозначения

- Векторы — столбцы; $x^\top y$ — скалярное произведение; $\|x\| = \sqrt{x^\top x}$ — евклидова норма.
- **Градиент** $\nabla f(x) \in \mathbb{R}^n$ — столбец частных производных.
- **Гессиан** $\nabla^2 f(x) \in \mathbb{R}^{n\times n}$ — симметричная матрица вторых производных.
- Матрица $A$ **положительно определена** ($A \succ 0$), если $v^\top A v > 0$ для всех $v \ne 0$; **положительно полуопределена** ($A \succeq 0$), если $v^\top A v \ge 0$. Для симметричной $A$ это равносильно тому, что все собственные числа положительны (соответственно неотрицательны).

Этого на сегодня достаточно. Два обозначения, которые понадобятся с лекции 4, — для справки:

<details>
<summary>Якобиан и формула Тейлора</summary>

- **Якобиан** вектор-функции $g$: $\dfrac{\partial g}{\partial x}(x) \in \mathbb{R}^{m \times n}$, строки — градиенты компонент $\nabla g_i(x)^\top$.
- Формула Тейлора второго порядка:

  $$f(x + p) = f(x) + \nabla f(x)^\top p + \tfrac12\, p^\top \nabla^2 f(x)\, p + o(\|p\|^2).$$

</details>

## 4. Основные определения

**Допустимое множество** (feasible set):

$$
\Omega = \{\, x \in \mathbb{R}^n : g(x) = 0,\; h(x) \ge 0 \,\}.
$$

Точка $x \in \Omega$ называется **допустимой**. Если $\Omega = \varnothing$, задача **несовместна** (infeasible).

Ограничение-неравенство $h_i$ **активно** в точке $x$, если $h_i(x) = 0$, и **неактивно**, если $h_i(x) > 0$. Множество индексов активных ограничений обозначим $\mathcal{A}(x)$. Равенства активны всегда.

<img src="img/01_feasible_set.png" width="480" alt="допустимое множество, активные и неактивные ограничения">

На картинке два неравенства вырезают из плоскости голубой угол, а равенство $g(x) = 0$ — кривую; допустимое множество $\Omega$ — их пересечение, оранжевый кусок кривой. Пунктирные окружности — линии уровня $f$: чем ближе к центру, тем меньше $f$. Решение $x^\ast$ — самая близкая к центру точка $\Omega$; в ней ограничение $h_2$ активно (мы «упёрлись» в него), а $h_1$ — нет.

**Определение (глобальный минимум).** Точка $x^\ast \in \Omega$ — глобальный минимум задачи (NLP), если $f(x^\ast) \le f(x)$ для всех $x \in \Omega$. Если неравенство строгое при $x \ne x^\ast$, минимум **строгий**.

**Определение (локальный минимум).** Точка $x^\ast \in \Omega$ — локальный минимум, если существует окрестность $\mathcal{N}$ точки $x^\ast$ такая, что $f(x^\ast) \le f(x)$ для всех $x \in \Omega \cap \mathcal{N}$. Аналогично определяется строгий локальный минимум.

<img src="img/02_minima_1d.png" width="560" alt="строгий, нестрогий и глобальный минимумы">

Значение $f(x^\ast)$ называют **оптимальным значением**, а точку $x^\ast$ — **решением** или **минимизатором** (minimizer). Различайте $\min f$ (число) и $\arg\min f$ (множество точек).

Несколько ситуаций, которые нужно уметь распознавать:

| Ситуация | Пример | Что происходит |
|----------|--------|----------------|
| Несовместна | $\min x$ при $x \ge 1,\; x \le 0$ | $\Omega = \varnothing$, решения нет |
| Не ограничена снизу | $\min x^3$ на $\mathbb{R}$ | $\inf f = -\infty$ |
| Инфимум не достигается | $\min e^{x}$ на $\mathbb{R}$ | $\inf f = 0$, но $e^x > 0$ всюду |
| Много минимумов | $\min (x^2 - 1)^2$ | глобальные минимумы $x = \pm 1$ |
| Локальный ≠ глобальный | $\min (x^2-1)^2 + 0.3x$ | два локальных минимума с разными значениями |

<img src="img/03_pathologies.png" width="560" alt="четыре патологии">

Последний случай — центральный для всего курса: большинство методов, которые мы будем изучать, **находят локальный минимум**, и результат зависит от начального приближения (пример 7.4).

## 5. Когда решение существует

**Теорема (Вейерштрасс).** Если $\Omega$ непусто и компактно (замкнуто и ограничено), а $f$ непрерывна на $\Omega$, то глобальный минимум существует.

Компактности часто нет — например, в безусловной задаче $\Omega = \mathbb{R}^n$. Тогда её заменяет рост функции на бесконечности.

**Определение.** Функция $f$ называется **коэрцитивной** на замкнутом множестве $\Omega$, если $f(x) \to +\infty$ при $\|x\| \to \infty$, $x \in \Omega$.

**Следствие.** Если $\Omega$ замкнуто и непусто, $f$ непрерывна и коэрцитивна на $\Omega$, то глобальный минимум существует.

<img src="img/04_coercive.png" width="600" alt="коэрцитивная и некоэрцитивная функции">

Идея на картинке: возьмём любую точку $x_0$ и посмотрим на множество $\{x : f(x) \le f(x_0)\}$ — «всё, что не хуже $x_0$». Если функция растёт на бесконечности, это множество ограничено, и минимум ищется на компакте. Для $e^x$ оно уходит в $-\infty$, и минимума нет.

<details>
<summary>Доказательство следствия (для любознательных)</summary>

Возьмём любую $x_0 \in \Omega$. Множество $\{x \in \Omega : f(x) \le f(x_0)\}$ замкнуто (как прообраз замкнутого при непрерывном отображении, пересечённый с замкнутым $\Omega$) и ограничено (иначе нашлась бы последовательность с $\|x_k\|\to\infty$ и $f(x_k) \le f(x_0)$, что противоречит коэрцитивности). По Вейерштрассу минимум на этом множестве достигается; он же является минимумом на всём $\Omega$. $\square$

</details>

Пример: $f(x) = \tfrac12 \|Ax - b\|^2$ коэрцитивна тогда и только тогда, когда $A$ имеет полный столбцовый ранг.

**Единственность** в общем случае не гарантирована (см. $(x^2-1)^2$). Основной инструмент, который её даёт, — строгая выпуклость; о ней на следующей лекции.

## 6. Классы задач

Класс задачи определяет и то, какой метод применим, и то, насколько трудно найти решение. Задачи классифицируют по нескольким независимым признакам.

### 6.1. По виду функций

| Класс | Целевая функция | Ограничения |
|-------|-----------------|-------------|
| Линейное программирование (LP) | $c^\top x$ | $Ax = b$, $Cx \ge d$ (линейные) |
| Квадратичное программирование (QP) | $\tfrac12 x^\top Q x + c^\top x$ | линейные |
| Квадратичное с квадратичными ограничениями (QCQP) | квадратичная | квадратичные |
| Нелинейное программирование (NLP) | произвольная гладкая | произвольные гладкие |

Каждый класс вкладывается в следующий: LP $\subset$ QP $\subset$ QCQP $\subset$ NLP. Чем уже класс, тем эффективнее специализированные методы: LP с миллионами переменных решаются рутинно, а для NLP общего вида и сотня переменных может оказаться трудной.

Отдельная ось — **структура** целевой функции. Задача наименьших квадратов $\min \tfrac12\Vert r(x)\Vert^2$ — это NLP (или QP, если $r$ линейна), но сумма квадратов позволяет строить специальные методы (Гаусс–Ньютон, лекция 5). Структурные признаки — сумма квадратов, разреженность, сепарабельность — не меняют класс, но меняют алгоритм.

### 6.2. По наличию ограничений

- **Безусловная** (unconstrained): $\Omega = \mathbb{R}^n$. Часть II курса.
- **С ограничениями-равенствами**. Часть III.
- **С ограничениями-неравенствами** (общий случай). Часть IV.

### 6.3. Выпуклые и невыпуклые

Задача **выпукла**, если $f$ выпукла и $\Omega$ — выпуклое множество (например, $g$ аффинна, а компоненты $h$ вогнуты). Точные определения — на лекции 2; сейчас важно главное свойство: для выпуклых задач **любой локальный минимум глобален**. Это главный водораздел в оптимизации:

> «Великий водораздел в оптимизации проходит не между линейностью и нелинейностью, а между выпуклостью и невыпуклостью.» — R. T. Rockafellar

<img src="img/05_classes.png" width="600" alt="вложенные классы и водораздел выпуклости">

LP всегда выпукла, QP выпукла при $Q \succeq 0$. Разница в поведении методов видна на картинке: в выпуклой задаче все старты приходят в одну точку, в невыпуклой — каждый в свою.

<img src="img/06_convex_vs_nonconvex.png" width="640" alt="выпуклая и невыпуклая функции: траектории из разных стартов">

**Как распознать выпуклость уже сейчас** (строгие определения и критерии — на лекции 2; этого хватит для домашнего задания):

- LP — всегда выпукла.
- QP выпукла тогда и только тогда, когда $Q \succeq 0$; проверка — собственные числа $Q$ (`np.linalg.eigvalsh`). Если есть отрицательное собственное число, задача невыпукла.
- Линейный МНК $\tfrac12\|Ax - b\|^2$ — выпуклая QP с $Q = A^\top A \succeq 0$.
- Линейные равенства и неравенства, шар $\|x\| \le r$, полуплоскости задают выпуклые множества; их пересечение выпукло.
- Нелинейное **равенство** $g(x) = 0$ (окружность, сфера) почти всегда делает допустимое множество невыпуклым: отрезок между двумя точками окружности в неё не попадает.
- Нелинейный МНК (модель нелинейна по параметрам) — как правило, невыпуклая задача.

### 6.4. Гладкие и негладкие

Мы работаем с гладкими ($C^2$) функциями — это позволяет использовать производные. Негладкие функции ($|x|$, $\max$, нормы $\|\cdot\|_1$, $\|\cdot\|_\infty$) часто удаётся переформулировать в гладкий вид добавлением переменных; см. раздел 8.

### 6.5. Непрерывные и дискретные

Если часть переменных обязана быть целой ($x_i \in \mathbb{Z}$ или $\{0,1\}$), получаем **целочисленное** или **смешанно-целочисленное** программирование (MILP, MINLP). Такие задачи в общем случае NP-трудны и решаются методами ветвей и границ, которые многократно решают непрерывные релаксации. В курсе они не рассматриваются, но всё, что мы изучим, — их «строительные блоки».

### 6.6. Что ещё бывает

Одной фразой, чтобы вы узнавали эти слова: в **оптимальном управлении** переменная — функция времени, а ограничение — дифференциальное уравнение; численно такие задачи дискретизируют и сводят к большим разреженным NLP (лекция 15). В **стохастической** оптимизации данные случайны и минимизируют математическое ожидание — так обучают нейросети. Наконец, **размер и структура** задачи ($n \sim 10$ или $10^6$, разреженные или плотные матрицы) определяют, какие операции линейной алгебры мы можем себе позволить; хороший солвер использует структуру, плохая формулировка её разрушает.

## 7. Четыре примера

Для каждого примера: постановка → стандартная форма → класс задачи.

### 7.1. Линейный метод наименьших квадратов

Даны измерения $(t_i, y_i)$, $i = 1,\dots,N$, и модель, линейная по параметрам $x \in \mathbb{R}^n$:

$$
\varphi(t; x) = x_1 \phi_1(t) + \dots + x_n \phi_n(t), \qquad\text{например } \varphi(t;x) = x_1 + x_2 t + x_3 t^2 .
$$

Ищем $x$, минимизирующий сумму квадратов невязок:

$$
\min_{x} \; \tfrac12 \sum_{i=1}^N \big(\varphi(t_i;x) - y_i\big)^2 = \tfrac12 \|Ax - y\|^2, \qquad A_{ij} = \phi_j(t_i).
$$

Раскрывая скобки, получаем $\tfrac12 x^\top (A^\top A) x - (A^\top y)^\top x + \text{const}$ — **безусловная выпуклая QP**. Градиент равен $A^\top(Ax - y)$; приравнивая его к нулю, получаем **нормальные уравнения**

$$
A^\top A\, x = A^\top y .
$$

Решение единственно, если $A$ полного столбцового ранга. На практике нормальные уравнения решать не стоит: обусловленность $A^\top A$ равна квадрату обусловленности $A$. Используют QR-разложение (`numpy.linalg.lstsq`).

<img src="img/07_lstsq.png" width="480" alt="подгонка полинома">

Полином 15-й степени проходит ближе к точкам, но «ловит шум» — это переобучение, и о нём задача 3(а) домашнего задания. Если модель нелинейна по параметрам (например, $x_1 e^{-x_2 t}$), идея та же, но задача становится невыпуклой — нелинейный МНК, лекция 5. Демо: часть (a).

### 7.2. Планирование производства (LP)

Завод выпускает два продукта в количествах $x_1, x_2$ с прибылью 3 и 5 за единицу. Три цеха имеют ограниченные мощности: $x_1 \le 4$, $2x_2 \le 12$, $3x_1 + 2x_2 \le 18$. Задача

$$
\max_{x} \; 3x_1 + 5x_2 \quad\text{при}\quad x_1 \le 4,\; 2x_2 \le 12,\; 3x_1 + 2x_2 \le 18,\; x \ge 0
$$

— **LP**. В стандартной форме: $\min (-3x_1 - 5x_2)$, $h(x) = (4 - x_1,\; 12 - 2x_2,\; 18 - 3x_1 - 2x_2,\; x_1,\; x_2) \ge 0$.

<img src="img/08_lp.png" width="420" alt="геометрия LP">

Допустимое множество — многоугольник, линии уровня прибыли — параллельные прямые. Сдвигаем прямую в сторону роста прибыли, пока она не выйдет из многоугольника: последняя точка касания — вершина $x^\ast = (2, 6)$, прибыль $36$. Активны ограничения 2 и 3. Демо: часть (b).

### 7.3. Подвешенная цепь

Цепь из $N$ грузов массы $m$, соединённых пружинами жёсткости $D$ (нулевой длины покоя), концы закреплены в точках $(-2, 1)$ и $(2, 1)$. Положение груза $i$ — $(y_i, z_i)$. Равновесие — минимум потенциальной энергии (упругая плюс гравитационная):

$$
\min_{y, z \in \mathbb{R}^N} \; \tfrac12 D \sum_{i=0}^{N} \big[(y_{i+1} - y_i)^2 + (z_{i+1} - z_i)^2\big] + m g \sum_{i=1}^{N} z_i ,
$$

где $(y_0,z_0)$ и $(y_{N+1},z_{N+1})$ — закреплённые концы. Энергия квадратична по переменным, ограничений нет: **безусловная выпуклая QP** с $2N$ переменными. Добавим «пол» — наклонную плоскость, ниже которой грузы опускаться не могут: $z_i \ge 0.5 + 0.1\, y_i$. Получаем **QP с линейными неравенствами**; часть ограничений активна (цепь лежит на полу), часть — нет.

<img src="img/09_chain.png" width="480" alt="подвешенная цепь без ограничений и с полом">

Этот пример будет сопровождать нас весь курс. Демо: часть (c) — там стандартная форма выписана сначала для одного груза (две переменные, как в разминке), потом для трёх, и только затем для сорока.

### 7.4. Невыпуклая функция одной переменной

Функция $f(x) = (x^2 - 1)^2 + 0.3x$ имеет два локальных минимума с разными значениями. Задача $\min_x f(x)$ — безусловная невыпуклая NLP, проще не бывает. Запустим стандартный метод (BFGS из `scipy.optimize.minimize`) из трёх начальных точек:

<img src="img/10_nonconvex_starts.png" width="480" alt="результат зависит от начальной точки">

Из $x_0 = -1.5$ метод приходит в глобальный минимум, из $x_0 = 0.3$ и $1.5$ — в локальный. Метод не ошибается: он честно находит *какой-то* минимум, а какой — зависит от старта. Демо: часть (d).

### 7.5. Где ещё

- **Портфель Марковица**: распределить капитал между активами так, чтобы риск (дисперсия доходности) был минимален при заданной ожидаемой доходности. Выпуклая QP с линейными ограничениями.
- **Оптимальное управление**: выбрать управляющие воздействия на каждом шаге, чтобы система пришла в цель с минимальными затратами. Большая, но очень разреженная NLP; при линейной динамике — QP. Именно такие задачи решают в MPC сотни раз в секунду (лекция 15).
- **Обучение модели**: логистическая регрессия — выпуклая гладкая безусловная задача, только переменных могут быть миллионы. Методы курса (L-BFGS) применимы напрямую.

## 8. Что значит «решить задачу численно»

**Итерационные методы.** Почти все алгоритмы курса строят последовательность $x_0, x_1, x_2, \dots$ по правилу $x_{k+1} = x_k + p_k$, где шаг $p_k$ вычисляется из информации о $f$, $g$, $h$ и их производных в точке $x_k$ (и, возможно, в предыдущих).

<img src="img/11_iterates.png" width="640" alt="итерации и критерий остановки">

Три вопроса про любой метод:

1. **Сходится ли** последовательность и к чему (стационарная точка, локальный минимум, глобальный)?
2. **Как быстро** сходится: сколько итераций нужно, чтобы получить ещё одну верную цифру? Точные определения скоростей — на лекции 4.
3. **Сколько стоит одна итерация**: вычисления функций, производных, решение линейной системы размера $n$?

**Локальность.** Методы, использующие производные, находят точку, в которой выполнены **условия оптимальности** (лекция 4) — обычно локальный минимум. Какой именно, зависит от $x_0$ (пример 7.4). Для выпуклых задач этой проблемы нет.

**Критерий остановки.** Точное решение мы не получим; останавливаемся, когда невязка условий оптимальности мала. Для безусловной задачи это $\|\nabla f(x_k)\| \le \varepsilon$ — правая часть картинки выше; аналог для задач с ограничениями появится на лекции 10. Полезно контролировать и $\|x_{k+1} - x_k\|$, и число итераций. Производные при этом лучше вычислять точно, а не конечными разностями — как, обсудим на лекции 7.

**Моделирование влияет на трудность.** Рассмотрим задачу $\min_x \sum_i |a_i^\top x - b_i|$ (подгонка в норме $\ell_1$, устойчивая к выбросам). Функция негладкая, и методы курса напрямую неприменимы. Введём вспомогательные переменные $s_i \ge |a_i^\top x - b_i|$:

$$
\min_{x, s} \; \sum_i s_i \quad\text{при}\quad -s_i \le a_i^\top x - b_i \le s_i .
$$

<img src="img/12_slack_l1.png" width="420" alt="вспомогательная переменная s ≥ |t|">

Множество $s \ge |t|$ — «стакан» над графиком модуля, и задаётся оно двумя линейными неравенствами. Минимизация $s$ прижимает точку к границе, поэтому в решении $s_i = |a_i^\top x - b_i|$, и ничего не потеряно. Получили **LP** — задачу, для которой есть надёжные и быстрые методы. Такие переформулировки — важнейший навык, и мы будем тренировать его в домашних заданиях.

**Готовые солверы vs собственная реализация.** В курсе мы делаем и то, и другое. Собственная реализация нужна, чтобы понимать, что происходит внутри, и уметь диагностировать сбои. Готовые солверы (SciPy, IPOPT, OSQP) — для реальных задач: в них вложены годы инженерной работы по устойчивости и обработке вырожденных случаев.

## 9. Итоги лекции

1. Задача оптимизации — это переменные, критерий и ограничения. Всё записываем в стандартной форме (NLP): $\min f(x)$ при $g(x) = 0$, $h(x) \ge 0$.
2. Решение — точка допустимого множества $\Omega$. Различаем глобальный, локальный и строгий минимум; в точке решения часть неравенств активна ($h_i = 0$), часть нет.
3. Минимум существует, если $\Omega$ компактно (Вейерштрасс) или $f$ коэрцитивна. Единственность — отдельный вопрос; ответ на него даст выпуклость (лекция 2).
4. Класс задачи — LP $\subset$ QP $\subset$ QCQP $\subset$ NLP; с ограничениями или без; выпуклая или нет; гладкая или нет — определяет и метод, и трудность. Главный водораздел — выпуклость.
5. Численные методы итерационны и локальны: из старта $x_0$ они приходят в *какой-то* минимум поблизости и останавливаются, когда невязка условий оптимальности мала.
6. Моделирование — часть решения: вспомогательные переменные превращают модуль и максимум в линейные неравенства, а негладкую задачу — в LP.

## 10. Обзор курса и инструменты

Курс состоит из четырёх частей и следует структуре курса *Numerical Optimization* М. Диля (Университет Фрайбурга):

| Часть | Лекции | Содержание |
|-------|--------|------------|
| I. Основы | 1–3 | постановка, классы задач, выпуклость, двойственность |
| II. Безусловная оптимизация | 4–7 | условия оптимальности, градиент, Ньютон, квази-Ньютон, Гаусс–Ньютон, линейный поиск, trust region, вычисление производных |
| III. Ограничения-равенства | 8–9 | множители Лагранжа, Ньютон–Лагранж, обобщённый Гаусс–Ньютон |
| IV. Ограничения-неравенства | 10–14 | ККТ, LP, QP, active set, методы внутренней точки, SQP, параметрическая оптимизация |
| Приложения | 15 | оптимальное управление и MPC, нелинейный МНК |

Подробный календарь, правила ДЗ и проекта, оценивание — в [программе курса](../../syllabus.md).

**Инструменты.** Python 3.10+, NumPy, SciPy (`scipy.optimize`: `minimize`, `linprog`, `least_squares`), matplotlib, Jupyter. С лекции 7 — CasADi (алгоритмическое дифференцирование и интерфейс к IPOPT). Установка: `uv sync` в корне репозитория, запуск ноутбуков — `uv run jupyter lab` (подробности в [README](../../README.md)).

**Литература к лекции.** Diehl, гл. 1; Nocedal & Wright, гл. 1; Boyd & Vandenberghe, гл. 1 и 4.1. На русском: Поляк, гл. 1; Жадан, ч. 1, гл. 1.

## 11. Практика на занятии

Вторая половина пары, около 40 минут. Всё, что нужно, — ноутбук [`demo01.ipynb`](demo01.ipynb) и домашнее задание.

| Время | Что делаем |
|-------|-----------|
| 10 мин | `demo01.ipynb`, разминки 1–2 вживую: интерфейс `minimize`, что лежит в результате, зачем `jac`, как добавить ограничение и увидеть, что оно активно. |
| 10 мин | Упражнение 12.1 у доски: три задачи → стандартная форма → класс. По пункту на студента. |
| 10 мин | Упражнение 12.4: LP графически, затем часть (b) ноутбука — сверить вершину с `linprog`, найти активные ограничения по `slack`. |
| 10 мин | Часть (d) ноутбука: мультистарт; обсуждение 12.5. Затем открыть [`hw01.ipynb`](../../homeworks/hw01.ipynb), убедиться, что данные генерируются, и начать задачу 3(б). |

## 12. Упражнения

Для разбора на занятии и самостоятельно. Ответы — в [`exercises01.md`](exercises01.md).

**12.1.** Запишите в стандартной форме (NLP) и определите класс:

(а) $\max_{x \in \mathbb{R}^2} x_1 x_2$ при $x_1 + x_2 = 10$, $x \ge 0$;

(б) найти точку окружности $x_1^2 + x_2^2 = 1$, ближайшую к точке $(3, 4)$;

(в) $\min_{x \in \mathbb{R}^n} \|Ax - b\|_2$ при $\|x\|_2 \le 1$.

**12.2.** Приведите пример задачи, у которой (а) допустимое множество непусто и ограничено, но минимума нет — какое условие теоремы Вейерштрасса при этом нарушено, и почему в стандартной форме (NLP) с непрерывными $g$, $h$ так не бывает; (б) бесконечно много глобальных минимумов; (в) есть ровно один локальный минимум, но нет глобального.

**12.3.** Для функции $f(x) = \tfrac12 x^\top Q x + c^\top x$ с симметричной $Q$ выпишите $\nabla f$ и $\nabla^2 f$. Когда $\arg\min f$ — единственная точка?

**12.4.** В примере 7.2 решите LP графически: нарисуйте допустимый многоугольник и линии уровня целевой функции. Убедитесь, что решение $(2, 6)$ — вершина. Какие ограничения активны? Что изменится, если прибыль от первого продукта вырастет до 10?

**12.5.** Запустите `demo01.ipynb`, часть (d). Объясните, почему запуски из разных начальных точек дают разные ответы. Как найти глобальный минимум в одномерном случае? Почему этот приём не масштабируется на $n = 100$?

**Домашнее задание 1** — [`homeworks/hw01.ipynb`](../../homeworks/hw01.ipynb), срок сдачи 23 сентября.

## Вопросы для самопроверки

1. Что такое допустимое множество, активное ограничение, локальный и глобальный минимум?
2. Сформулируйте теорему Вейерштрасса и условие коэрцитивности. Приведите пример, когда минимум не достигается.
3. Чем LP отличается от QP, QP от NLP? Почему выпуклость важнее линейности?
4. Что такое нормальные уравнения и почему их не стоит решать «в лоб»?
5. Что означает «решить задачу» для итерационного метода? Какие три характеристики метода нас интересуют?
6. Как переформулировать негладкую задачу с $|\cdot|$ в гладкую?